# Data Processing – Seeds Dataset

Notebook này xử lý `seeds_dataset.csv` và tạo hai file:

- `seeds_train.csv`
- `seeds_test.csv`

Quy trình:

1. Đọc dữ liệu.
2. Chuẩn hóa tên cột.
3. Kiểm tra kiểu dữ liệu, missing values và duplicate.
4. Kiểm tra phân bố target.
5. Chia train/test theo tỷ lệ 80/20 với `stratify`.
6. Lưu hai file CSV.

> Chưa chuẩn hóa bằng `StandardScaler` ở notebook này. Scaling sẽ được thực hiện khi train mô hình để tránh data leakage.


In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 1. Khai báo đường dẫn

In [3]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("Dry_Bean_Dataset") / "Dry_Bean_Dataset.xlsx"

OUTPUT_DIR = DATA_PATH.parent
TRAIN_OUTPUT_PATH = OUTPUT_DIR / "dry_bean_train.csv"
TEST_OUTPUT_PATH = OUTPUT_DIR / "dry_bean_test.csv"

print("Đường dẫn:", DATA_PATH.resolve())
print("File tồn tại:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file tại: {DATA_PATH.resolve()}"
    )

Đường dẫn: C:\Users\LENOVO\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\Dry_Bean_Dataset.xlsx
File tồn tại: True


## 2. Đọc dữ liệu

In [4]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [5]:
df = pd.read_excel(
    DATA_PATH,
    engine="openpyxl"
)

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

print(df.shape)
print(df["class"].head())
print(df["class"].unique())

(13611, 17)
0    SEKER
1    SEKER
2    SEKER
3    SEKER
4    SEKER
Name: class, dtype: str
<StringArray>
['SEKER', 'BARBUNYA', 'BOMBAY', 'CALI', 'HOROZ', 'SIRA', 'DERMASON']
Length: 7, dtype: str


## 3. Chuẩn hóa tên cột

In [6]:
target = "class"

numeric_columns = [
    column for column in df.columns
    if column != target
]

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Làm sạch target dạng chữ
df[target] = (
    df[target]
    .astype(str)
    .str.strip()
    .str.upper()
)

## 4. Kiểm tra cấu trúc dữ liệu

In [7]:
print("Thông tin dữ liệu:")
df.info()

print("\nThống kê mô tả:")
display(df.describe(include="all").T)


Thông tin dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 13611 entries, 0 to 13610
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   area             13611 non-null  int64  
 1   perimeter        13611 non-null  float64
 2   majoraxislength  13611 non-null  float64
 3   minoraxislength  13611 non-null  float64
 4   aspectration     13611 non-null  float64
 5   eccentricity     13611 non-null  float64
 6   convexarea       13611 non-null  int64  
 7   equivdiameter    13611 non-null  float64
 8   extent           13611 non-null  float64
 9   solidity         13611 non-null  float64
 10  roundness        13611 non-null  float64
 11  compactness      13611 non-null  float64
 12  shapefactor1     13611 non-null  float64
 13  shapefactor2     13611 non-null  float64
 14  shapefactor3     13611 non-null  float64
 15  shapefactor4     13611 non-null  float64
 16  class            13611 non-null  str    
dtypes: f

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
area,13611.0,NaN,NaN,NaN,53048.284549,29324.095717,20420.0,36328.0,44652.0,61332.0,254616.0
perimeter,13611.0,NaN,NaN,NaN,855.283459,214.289696,524.736,703.5235,794.941,977.213,1985.37
majoraxislength,13611.0,NaN,NaN,NaN,320.141867,85.694186,183.601165,253.303633,296.883367,376.495012,738.860153
minoraxislength,13611.0,NaN,NaN,NaN,202.270714,44.970091,122.512653,175.84817,192.431733,217.031741,460.198497
aspectration,13611.0,NaN,NaN,NaN,1.583242,0.246678,1.024868,1.432307,1.551124,1.707109,2.430306
eccentricity,13611.0,NaN,NaN,NaN,0.750895,0.092002,0.218951,0.715928,0.764441,0.810466,0.911423
convexarea,13611.0,NaN,NaN,NaN,53768.200206,29774.915817,20684.0,36714.5,45178.0,62294.0,263261.0
equivdiameter,13611.0,NaN,NaN,NaN,253.06422,59.17712,161.243764,215.068003,238.438026,279.446467,569.374358
extent,13611.0,NaN,NaN,NaN,0.749733,0.049086,0.555315,0.718634,0.759859,0.786851,0.866195
solidity,13611.0,NaN,NaN,NaN,0.987143,0.00466,0.919246,0.98567,0.988283,0.990013,0.994677


## 5. Kiểm tra missing values

In [8]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

display(missing)
print("Tổng số missing values:", df.isna().sum().sum())


,missing_count,missing_percent
area,0,0.0
perimeter,0,0.0
majoraxislength,0,0.0
minoraxislength,0,0.0
aspectration,0,0.0
eccentricity,0,0.0
convexarea,0,0.0
equivdiameter,0,0.0
extent,0,0.0
solidity,0,0.0


Tổng số missing values: 0


In [9]:
rows_before_missing = len(df)

df = df.dropna().reset_index(drop=True)

rows_after_missing = len(df)

print("Số hàng trước khi bỏ missing:", rows_before_missing)
print("Số hàng sau khi bỏ missing:", rows_after_missing)
print("Số hàng đã bỏ:", rows_before_missing - rows_after_missing)


Số hàng trước khi bỏ missing: 13611
Số hàng sau khi bỏ missing: 13611
Số hàng đã bỏ: 0


## 6. Kiểm tra và xóa duplicate

In [10]:
duplicate_count = df.duplicated().sum()

print("Số dòng trùng hoàn toàn:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Kích thước sau khi xóa duplicate:", df.shape)


Số dòng trùng hoàn toàn: 68
Kích thước sau khi xóa duplicate: (13543, 17)


## 7. Xác định feature và target

In [23]:
target = "class"

# Tất cả các cột ngoại trừ target đều là feature
features = [
    column for column in df.columns
    if column != target
]

X_bean = df[features].copy()
y_bean = df[target].copy()

print("Danh sách feature:")
print(features)

print("\nSố feature:", len(features))
print("X shape:", X_bean.shape)
print("y shape:", y_bean.shape)

print("\nPhân bố target:")
print(y.value_counts())

Danh sách feature:
['area', 'perimeter', 'majoraxislength', 'minoraxislength', 'aspectration', 'eccentricity', 'convexarea', 'equivdiameter', 'extent', 'solidity', 'roundness', 'compactness', 'shapefactor1', 'shapefactor2', 'shapefactor3', 'shapefactor4']

Số feature: 16
X shape: (13543, 16)
y shape: (13543,)

Phân bố target:
class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1860
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


## 9. Kiểm tra phân bố target

In [12]:
print("Số lượng từng lớp:")
print(y.value_counts().sort_index())

print("\nTỷ lệ từng lớp:")
print(y.value_counts(normalize=True).sort_index().round(4))

print("\nCác nhãn có trong target:")
print(sorted(y.unique()))


Số lượng từng lớp:
class
BARBUNYA    1322
BOMBAY       522
CALI        1630
DERMASON    3546
HOROZ       1860
SEKER       2027
SIRA        2636
Name: count, dtype: int64

Tỷ lệ từng lớp:
class
BARBUNYA    0.0976
BOMBAY      0.0385
CALI        0.1204
DERMASON    0.2618
HOROZ       0.1373
SEKER       0.1497
SIRA        0.1946
Name: proportion, dtype: float64

Các nhãn có trong target:
['BARBUNYA', 'BOMBAY', 'CALI', 'DERMASON', 'HOROZ', 'SEKER', 'SIRA']


## 10. Chia train/test

In [24]:
X_train_bean, X_test_bean, y_train_bean, y_test_bean = train_test_split(
    X_bean,
    y_bean,
    test_size=0.2,
    random_state=42,
    stratify=y_bean
)

print("Train shape:", X_train_bean.shape)
print("Test shape:", X_test_bean.shape)

Train shape: (10834, 16)
Test shape: (2709, 16)


## 11. Ghép feature và target

In [25]:
train_df = X_train_bean.copy()
train_df[target] = y_train_bean

test_df = X_test_bean.copy()
test_df[target] = y_test_bean

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

display(train_df.head())


,area,perimeter,majoraxislength,minoraxislength,aspectration,eccentricity,convexarea,equivdiameter,extent,solidity,roundness,compactness,shapefactor1,shapefactor2,shapefactor3,shapefactor4,class
0,69471,1069.638,399.100245,225.005782,1.773733,0.825923,71088,297.410868,0.707386,0.977254,0.763027,0.745203,0.005745,0.001093,0.555328,0.985004,CALI
1,82877,1162.581,391.817013,270.836144,1.446694,0.722634,84171,324.841921,0.825986,0.984627,0.770544,0.829065,0.004728,0.001378,0.687349,0.994384,BARBUNYA
2,65042,1023.506,419.202858,198.962774,2.106941,0.880190,65748,287.774298,0.783403,0.989262,0.780231,0.686480,0.006445,0.000883,0.471255,0.992906,HOROZ
3,41315,758.920,287.438268,183.447580,1.566869,0.769858,41704,229.355383,0.791930,0.990672,0.901417,0.797929,0.006957,0.001740,0.636691,0.997611,SIRA
4,91088,1168.645,459.300729,253.950486,1.808623,0.833243,91799,340.553731,0.789051,0.992255,0.838119,0.741461,0.005042,0.000940,0.549765,0.994318,CALI


## 12. Kiểm tra phân bố lớp sau khi chia

In [26]:
print("Phân bố lớp trong train:")
print(train_df[target].value_counts().sort_index())

print("\nTỷ lệ lớp trong train:")
print(train_df[target].value_counts(normalize=True).sort_index().round(4))

print("\nPhân bố lớp trong test:")
print(test_df[target].value_counts().sort_index())

print("\nTỷ lệ lớp trong test:")
print(test_df[target].value_counts(normalize=True).sort_index().round(4))


Phân bố lớp trong train:
class
BARBUNYA    1057
BOMBAY       418
CALI        1304
DERMASON    2837
HOROZ       1488
SEKER       1621
SIRA        2109
Name: count, dtype: int64

Tỷ lệ lớp trong train:
class
BARBUNYA    0.0976
BOMBAY      0.0386
CALI        0.1204
DERMASON    0.2619
HOROZ       0.1373
SEKER       0.1496
SIRA        0.1947
Name: proportion, dtype: float64

Phân bố lớp trong test:
class
BARBUNYA    265
BOMBAY      104
CALI        326
DERMASON    709
HOROZ       372
SEKER       406
SIRA        527
Name: count, dtype: int64

Tỷ lệ lớp trong test:
class
BARBUNYA    0.0978
BOMBAY      0.0384
CALI        0.1203
DERMASON    0.2617
HOROZ       0.1373
SEKER       0.1499
SIRA        0.1945
Name: proportion, dtype: float64


## 13. Lưu thành hai file CSV

In [27]:
train_df.to_csv(TRAIN_OUTPUT_PATH, index=False)
test_df.to_csv(TEST_OUTPUT_PATH, index=False)

print("Đã lưu file train:", TRAIN_OUTPUT_PATH.resolve())
print("Đã lưu file test:", TEST_OUTPUT_PATH.resolve())


Đã lưu file train: C:\Users\LENOVO\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\dry_bean_train.csv
Đã lưu file test: C:\Users\LENOVO\mliot-pyml-2026-hw\week04\Homework_b7\Dry_Bean_Dataset\dry_bean_test.csv


## 14. Kiểm tra lại file đã lưu

In [28]:
saved_train = pd.read_csv(TRAIN_OUTPUT_PATH)
saved_test = pd.read_csv(TEST_OUTPUT_PATH)

print("Saved train shape:", saved_train.shape)
print("Saved test shape:", saved_test.shape)

print("\nMissing trong train:", saved_train.isna().sum().sum())
print("Missing trong test:", saved_test.isna().sum().sum())

print("\nCác cột trong train:")
print(saved_train.columns.tolist())


Saved train shape: (10834, 17)
Saved test shape: (2709, 17)

Missing trong train: 0
Missing trong test: 0

Các cột trong train:
['area', 'perimeter', 'majoraxislength', 'minoraxislength', 'aspectration', 'eccentricity', 'convexarea', 'equivdiameter', 'extent', 'solidity', 'roundness', 'compactness', 'shapefactor1', 'shapefactor2', 'shapefactor3', 'shapefactor4', 'class']


---
# Bài 1: Titanic Logistic Regression

## Bước 1: Xử lý dữ liệu giống BTVN trước.

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

Sẵn sàng.


In [33]:
try:
    df_titanic = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df_titanic = pd.read_csv(url)
    df_titanic.columns = [c.lower() for c in df_titanic.columns]
    print("Đã tải từ URL.")
df_titanic.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [34]:
print(df_titanic.isna().mean())
leaky = [
    "class",
    "who",
    "adult_male",
    "deck",
    "embark_town",
    "alive",
    "alone"
]      # điền danh sách cột cần bỏ (chỉ những cột có trong df)
df_titanic = df_titanic.drop(columns=leaky)
print(list(df_titanic.columns))

survived       0.000000
pclass         0.000000
sex            0.000000
age            0.198653
sibsp          0.000000
parch          0.000000
fare           0.000000
embarked       0.002245
class          0.000000
who            0.000000
adult_male     0.000000
deck           0.772166
embark_town    0.002245
alive          0.000000
alone          0.000000
dtype: float64
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']


In [38]:
def dem_outlier_iqr(s):
    s = s.dropna()
    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    IQR = Q3 -Q1
    botttom_outlier = Q1 - 1.5*IQR
    top_outlier = Q3+1.5*IQR
        # trả về số lượng outlier theo IQR
    return((s<botttom_outlier)| (s>top_outlier)).sum()
def dem_outlier_zscore(s, nguong=3.0):
    s = s.dropna()
    z = np.abs(stats.zscore(s))
        # trả về số lượng outlier theo Z-score
    return (z>nguong).sum()
# for col in ["age", "fare"]:
for col in ["age", "fare"]:
    print(f"{col}")
    print("IQR:", dem_outlier_iqr(df_titanic[col]))
    print("Z_score:", dem_outlier_zscore(df_titanic[col]))

age
IQR: 11
Z_score: 2
fare
IQR: 116
Z_score: 20


## Bước 2: Chia tập train và test giống tỉ lệ trước.

In [40]:
X_titanic = df_titanic.drop(columns="survived")
y_titanic = df_titanic["survived"]

X_tmp_titanic, X_test_titanic, y_tmp_titanic, y_test_titanic = train_test_split(
    X_titanic,
    y_titanic,
    test_size=0.15,
    stratify = y_titanic
)
X_train_titanic, X_val_titanic, y_train_titanic, y_val_titanic = train_test_split(
    X_tmp_titanic,
    y_tmp_titanic,
    test_size=15/85,
    stratify=y_tmp_titanic
)
print("Train:", X_train_titanic.shape, y_train_titanic.shape)
print("Validation:", X_val_titanic.shape, y_val_titanic.shape)
print("Test:", X_test_titanic.shape, y_test_titanic.shape)
# in tỷ lệ survived từng tập
print(f"Train: {y_train_titanic.mean()*100:.3f}")
print(f"Validation: {y_val_titanic.mean()*100:.3f}")
print(f"Test: {y_test_titanic.mean()*100:.3f}")

Train: (623, 7) (623,)
Validation: (134, 7) (134,)
Test: (134, 7) (134,)
Train: 38.363
Validation: 38.806
Test: 38.060


## Bước 3: Pipeline.

In [41]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_so  = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", pipe_so, num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols)
])

# preprocess.fit(X_train)
preprocess.fit(X_train_titanic)               # # fit CHỈ trên train
X_train_t = preprocess.transform(X_train_titanic)
# ... transform cho val, test
X_val_t = preprocess.transform(X_val_titanic)
X_test_t = preprocess.transform(X_test_titanic)
print(X_train_t.shape, list(preprocess.get_feature_names_out()))

(623, 10) ['num__age', 'num__sibsp', 'num__parch', 'num__fare', 'cat__sex_female', 'cat__sex_male', 'cat__embarked_C', 'cat__embarked_Q', 'cat__embarked_S', 'ord__pclass']


## Bước 4: Linear Regression và Logistic Regression.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_t, y_train_titanic)
y_pred_lr_score = lr_model.predict(X_test_t)
y_pred_lr = (y_pred_lr_score >= 0.5).astype(int)
TP = np.sum((y_test_titanic == 1) & (y_pred_lr == 1))
TN = np.sum((y_test_titanic == 0) & (y_pred_lr == 0))
FP = np.sum((y_test_titanic == 0) & (y_pred_lr == 1))
FN = np.sum((y_test_titanic == 1) & (y_pred_lr == 0))
accuracy_lr = (TP + TN) / len(y_test_titanic)
precision_lr = TP / (TP + FP)
recall_lr = TP / (TP + FN)
f1_lr = 2 * precision_lr * recall_lr / (precision_lr + recall_lr)
print(f"Accuracy : {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"Recall   : {recall_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")


log_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)
log_model.fit(X_train_t, y_train_titanic)
y_pred_log = log_model.predict(X_test_t)
TP = np.sum((y_test_titanic == 1) & (y_pred_log == 1))
TN = np.sum((y_test_titanic == 0) & (y_pred_log == 0))
FP = np.sum((y_test_titanic == 0) & (y_pred_log == 1))
FN = np.sum((y_test_titanic == 1) & (y_pred_log == 0))
accuracy_log = (TP + TN) / len(y_test_titanic)
precision_log = TP / (TP + FP)
recall_log = TP / (TP + FN)
f1_log = 2 * precision_log * recall_log / (precision_log + recall_log)
print("-----------------")
print(f"Accuracy : {accuracy_log:.4f}")
print(f"Precision: {precision_log:.4f}")
print(f"Recall   : {recall_log:.4f}")
print(f"F1-score : {f1_log:.4f}")

Accuracy : 0.8209
Precision: 0.8000
Recall   : 0.7059
F1-score : 0.7500
-----------------
Accuracy : 0.8209
Precision: 0.8000
Recall   : 0.7059
F1-score : 0.7500
Linear
[[np.int64(74), np.int64(9)], [np.int64(15), np.int64(36)]]


## Bước 5: Nhận xét Linear và Logistic:

- Ta thấy Linear Regression đạt Accuracy (82.09%), Precision (80.00%) và F1-score (75.00%) giống hệt Logistic Regression. Recall của hai mô hình đều bằng 70.59%.
Vì vậy nên chưa thể kết luận Linear Regression tốt hơn chỉ dựa trên một lần chia dữ liệu. Kết quả có thể thay đổi khi sử dụng cách chia dữ liệu khác hoặc đánh giá bằng Cross-Validation. Logistic Regression được thiết kế dành riêng cho bài toán phân loại nhị phân. Mô hình dự đoán xác suất một hành khách sống sót và đưa ra nhãn phân loại dựa trên xác suất đó. Trong khi đó, Linear Regression được thiết kế cho bài toán hồi quy, đầu ra là giá trị liên tục nên cần đặt thêm ngưỡng (0.5) để chuyển thành nhãn 0 hoặc 1. Do đó, mặc dù trong thí nghiệm này Linear Regression và Logistic Regression có kết quả giống nhau về các chỉ số đánh giá, Logistic Regression vẫn là mô hình phù hợp hơn đối với bài toán dự đoán hành khách sống sót vì đúng với bản chất của bài toán phân loại.

---
# Bài 2: Processing_seeds

## 1) Logistic Regression:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)
log_model = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(random_state=42, max_iter=5000))
])
log_model.fit(X_train_bean, y_train_bean)
y_pred_log = log_model.predict(X_test_bean)
accuracy_log = accuracy_score(y_test_bean, y_pred_log)
precision_log = precision_score(y_test_bean, y_pred_log, average="weighted")
recall_log = recall_score(y_test_bean, y_pred_log, average="weighted")
f1_log = f1_score(y_test_bean, y_pred_log, average="weighted")
cm_log = confusion_matrix(y_test_bean, y_pred_log)

print(f"Accuracy : {accuracy_log:.4f}")
print(f"Precision: {precision_log:.4f}")
print(f"Recall   : {recall_log:.4f}")
print(f"F1-score : {f1_log:.4f}")
print("\nConfusion Matrix:")
print(cm_log)

Accuracy : 0.9192
Precision: 0.9197
Recall   : 0.9192
F1-score : 0.9193

Confusion Matrix:
[[236   0  18   0   0   4   7]
 [  0 104   0   0   0   0   0]
 [  8   0 307   0   5   2   4]
 [  0   0   0 643   0  13  53]
 [  1   0  11   5 351   0   4]
 [  9   0   0   5   0 383   9]
 [  1   0   1  41   8  10 466]]


## 2) KNN:

In [67]:
knn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
knn_model.fit(X_train_bean, y_train_bean)
y_pred_knn = knn_model.predict(X_test_bean)
accuracy_knn = accuracy_score(y_test_bean, y_pred_knn)
precision_knn = precision_score(y_test_bean, y_pred_knn, average="weighted")
recall_knn = recall_score(y_test_bean, y_pred_knn, average="weighted")
f1_knn = f1_score(y_test_bean, y_pred_knn, average="weighted")
cm_knn = confusion_matrix(y_test_bean, y_pred_knn)

print(f"Accuracy : {accuracy_knn:.4f}")
print(f"Precision: {precision_knn:.4f}")
print(f"Recall   : {recall_knn:.4f}")
print(f"F1-score : {f1_knn:.4f}")
print("\nConfusion Matrix:")
print(cm_knn)

Accuracy : 0.9155
Precision: 0.9163
Recall   : 0.9155
F1-score : 0.9157

Confusion Matrix:
[[232   0  21   0   1   3   8]
 [  0 104   0   0   0   0   0]
 [  8   0 309   0   5   2   2]
 [  0   0   0 647   0  10  52]
 [  0   0  13   4 347   0   8]
 [  6   0   1   7   0 380  12]
 [  3   0   0  49   8   6 461]]


## 3) So sánh 2 cách:

In [68]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Logistic Regression": [
        accuracy_log,
        precision_log,
        recall_log,
        f1_log
    ],
    "KNN": [
        accuracy_knn,
        precision_knn,
        recall_knn,
        f1_knn
    ]
})
comparison

,Metric,Logistic Regression,KNN
0,Accuracy,0.919158,0.915467
1,Precision,0.919726,0.916315
2,Recall,0.919158,0.915467
3,F1-score,0.919290,0.915652
